# 01 — Stitch Transcripts (Bronze → Silver)

**What this does:** Transforms line-by-line Genesys transcripts into full conversations with metadata for AI scoring.

```
BEFORE (transcripts_raw)                                                    AFTER (silver_conversations)
+----------------+------+----------+------------------+-----------------+   +----------------+--------------+---------+------------------+-------------------------------------+
| interaction_id | line | who      | date_time        | text            |   | interaction_id | agent        | queue   | start / end      | full_transcript                     |
+----------------+------+----------+------------------+-----------------+   +----------------+--------------+---------+------------------+-------------------------------------+
| call-001       | 1    | AGENT    | 2026-05-11 07:24 | Hi, this is...  |   | call-001       | Maria Garcia | Billing | 07:24 -> 07:29   | [07:24:00] AGENT: Hi, this is...    |
| call-001       | 2    | CUSTOMER | 2026-05-11 07:24 | I have a...     |   |                |              |         | duration: 355s   | [07:24:29] CUSTOMER: I have a...    |
| call-001       | 3    | AGENT    | 2026-05-11 07:25 | Let me...       |   |                |              |         |                  | [07:24:59] AGENT: Let me...         |
| call-001       | 4    | CUSTOMER | 2026-05-11 07:25 | Thanks          |   |                |              |         |                  | [07:25:28] CUSTOMER: Thanks         |
| call-002       | 1    | AGENT    | 2026-05-29 18:29 | Welcome...      |   | call-002       | James Smith  | Nurse   | 18:29 -> 18:40   | [18:29:00] AGENT: Welcome to...     |
| call-002       | 2    | CUSTOMER | 2026-05-29 18:30 | My child...     |   |                |              |         | duration: 649s   | [18:29:38] CUSTOMER: My child...    |
| ...            | ...  | ...      | ...              | ...             |   | ...            | ...          | ...     | ...              | ...                                 |
+----------------+------+----------+------------------+-----------------+   +----------------+--------------+---------+------------------+-------------------------------------+
  500+ rows (2-17 per call)                                                   50 rows (1 per call, timestamps inline)
```

**Why this step is needed:** Genesys Cloud exports transcripts as one row per utterance (confirmed by the `line_num` bigint column in `dev_ds_b_playground_hackathon_call_transcripts`). AI scoring needs the full conversation as a single text block.

**Key operations:**
1. GROUP BY `interaction_id` + ORDER BY `line_num` -- reassemble utterances in order
2. CONCAT with speaker labels (+ optional per-utterance timestamps if `date_time` is available)
3. JOIN to `interactions_raw` on `Conversation_ID` -- attach metadata (queue, agent, duration)

> **Note on ordering:** The stitch uses `line_num` (bigint) for sequential utterance ordering -- NOT `date_time`. This means the stitching works correctly even if per-utterance timestamps are missing. The `date_time` field is purely optional enrichment embedded in the output text. If `date_time` is NULL, timestamps are omitted and the transcript shows speaker labels only.

**Rubric Criterion 1:** End-to-End Functionality — "Fully automated ingest → AI → validate → query"

---
*Prerequisite: `transcripts_raw` and `interactions_raw` must exist in Delta (run `00_Generate_Data` for synthetic data, or point to your Genesys export)*

In [0]:
%sql
-- Each row is ONE utterance from a call. We need to stitch these into full conversations.
SELECT interaction_id, line_num, date_time, participant, participant_type, transcribed_text
FROM mmt_aws_usw2_catalog.contact_calls.transcripts_raw
WHERE interaction_id = (SELECT interaction_id FROM mmt_aws_usw2_catalog.contact_calls.transcripts_raw LIMIT 1)
ORDER BY line_num

interaction_id,line_num,date_time,participant,participant_type,transcribed_text
d2c75d2a-88a7-457c-aeb3-82c5bbec5ea2,1,2026-05-11 07:24:00,Robert Park,internal,"HCP referrals, this is Robert Park. How can I help you?"
d2c75d2a-88a7-457c-aeb3-82c5bbec5ea2,2,2026-05-11 07:24:29,Customer,external,I need a referral to see a endocrinologist. My doctor said they would send one but I haven't heard anything.
d2c75d2a-88a7-457c-aeb3-82c5bbec5ea2,3,2026-05-11 07:24:59,Robert Park,internal,I can look into that. May I have your name and date of birth?
d2c75d2a-88a7-457c-aeb3-82c5bbec5ea2,4,2026-05-11 07:25:28,Customer,external,"Linda Park, 03/21/1997."
d2c75d2a-88a7-457c-aeb3-82c5bbec5ea2,5,2026-05-11 07:25:58,Robert Park,internal,The referral was approved. Let me get you scheduled.
d2c75d2a-88a7-457c-aeb3-82c5bbec5ea2,6,2026-05-11 07:26:27,Customer,external,Sounds good.
d2c75d2a-88a7-457c-aeb3-82c5bbec5ea2,7,2026-05-11 07:26:57,Customer,external,Can you also check on my other appointment?
d2c75d2a-88a7-457c-aeb3-82c5bbec5ea2,8,2026-05-11 07:27:27,Robert Park,internal,"Sure, let me pull that up."
d2c75d2a-88a7-457c-aeb3-82c5bbec5ea2,9,2026-05-11 07:27:56,Robert Park,internal,I see it here. Everything looks confirmed.
d2c75d2a-88a7-457c-aeb3-82c5bbec5ea2,10,2026-05-11 07:28:26,Robert Park,internal,I'll make sure this gets resolved. You should hear back within 24 hours. Anything else?


### The Transform

The cell below takes those individual utterances (2-17 per call) and stitches them into **one row** with the full conversation:

```
                   transcripts_raw (500+ rows)             interactions_raw (50 rows)
                         |                                        |
                         |  GROUP BY interaction_id               |
                         |  ORDER BY line_num                     |
                         |  CONCAT_WS with speaker labels         |
                         |  PRESERVE start/end timestamps         |
                         |                                        |
                         v                                        |
               stitched (50 rows)                                 |
               - full_transcript                                  |
               - transcript_start_time                            |
               - transcript_end_time                              |
               - transcript_duration                              |
               - num_utterances                                   |
                         |                                        |
                         +--------------- JOIN -------------------+
                                          |              (on interaction_id = Conversation_ID)
                                          v
                            silver_conversations (50 rows)
                            - full_transcript (complete text)
                            - transcript_start_time, end_time, duration
                            - agent_name, queue, division
                            - call_duration_seconds
                            - skills, language, wrap_up
```

In [0]:
%sql
-- Core ETL: Group utterances by call, order by line_num (NOT date_time), concatenate into full transcript
-- line_num is the guaranteed sequential ordering column from Genesys; date_time is optional enrichment
CREATE OR REPLACE TABLE mmt_aws_usw2_catalog.contact_calls.silver_conversations AS
WITH stitched AS (
  SELECT 
    t.interaction_id,
    t.direction,
    t.transcript_duration,
    t.transcript_start_time,
    t.transcript_end_time,
    -- Concatenate all lines into one conversation with speaker labels
    CONCAT_WS('\n', 
      COLLECT_LIST(
        -- Timestamps: included if date_time is available; gracefully omits if NULL
        CONCAT(
          CASE WHEN t.date_time IS NOT NULL THEN CONCAT('[', t.date_time, '] ') ELSE '' END,
          UPPER(t.participant_type), ': ', t.transcribed_text
        )
      )
    ) AS full_transcript,
    COUNT(*) AS num_utterances
  FROM (
    SELECT * FROM mmt_aws_usw2_catalog.contact_calls.transcripts_raw ORDER BY interaction_id, line_num
  ) t
  GROUP BY t.interaction_id, t.direction, t.transcript_duration, 
           t.transcript_start_time, t.transcript_end_time
)
SELECT 
  s.*,
  i.Users AS agent_name,
  i.Queue AS queue,
  i.Duration AS call_duration_seconds,
  i.Skills AS skills,
  i.Languages AS language,
  i.Wrap_up AS wrap_up,
  i.Division AS division,
  i.Conversation_ID
FROM stitched s
JOIN mmt_aws_usw2_catalog.contact_calls.interactions_raw i ON s.interaction_id = i.Conversation_ID;

-- Show results immediately so the output isn't empty
SELECT
  COUNT(*) AS conversations_created,
  SUM(num_utterances) AS utterances_stitched,
  ROUND(AVG(call_duration_seconds), 0) AS avg_call_duration_sec
FROM mmt_aws_usw2_catalog.contact_calls.silver_conversations

conversations_created,utterances_stitched,avg_call_duration_sec
50,517,297.0


In [0]:
%sql
-- Distribution stats: how varied are the conversations?
SELECT 
  COUNT(*) AS total_conversations,
  SUM(num_utterances) AS total_utterances_stitched,
  MIN(num_utterances) AS shortest_call_lines,
  MAX(num_utterances) AS longest_call_lines,
  ROUND(AVG(num_utterances), 1) AS avg_lines_per_call,
  ROUND(AVG(call_duration_seconds), 0) AS avg_duration_sec,
  COUNT(CASE WHEN num_utterances <= 5 THEN 1 END) AS abandoned_short_calls,
  COUNT(CASE WHEN num_utterances >= 12 THEN 1 END) AS extended_calls
FROM mmt_aws_usw2_catalog.contact_calls.silver_conversations

total_conversations,total_utterances_stitched,shortest_call_lines,longest_call_lines,avg_lines_per_call,avg_duration_sec,abandoned_short_calls,extended_calls
50,517,2,17,10.3,297.0,6,17


In [0]:
%sql
-- Sample output: one row per call with full transcript text
SELECT 
  interaction_id,
  agent_name,
  queue,
  transcript_start_time,
  transcript_end_time,
  transcript_duration,
  num_utterances,
  LEFT(full_transcript, 300) AS transcript_preview
FROM mmt_aws_usw2_catalog.contact_calls.silver_conversations
ORDER BY num_utterances DESC
LIMIT 5

interaction_id,agent_name,queue,transcript_start_time,transcript_end_time,transcript_duration,num_utterances,transcript_preview
b1743b2e-5130-4412-8aca-300b0075a7f5,Amy Martinez,Nurse Advice,2026-05-20 15:52:00,2026-05-20 15:56:50,290,17,"[2026-05-20 15:52:00] INTERNAL: HCP nurse advice line, this is Amy Martinez. Before we begin, may I have your name and date of birth for verification? [2026-05-20 15:52:17] EXTERNAL: It's Michael Chen, date of birth 04/21/1967. [2026-05-20 15:52:34] INTERNAL: Thank you. What symptoms are you experie"
2d3db05e-33c8-488b-96ac-f4367b5a5700,Carlos Rodriguez,Nurse Advice,2026-05-03 11:52:00,2026-05-03 11:57:26,326,16,"[2026-05-03 11:52:00] INTERNAL: HCP nurse advice line, this is Carlos Rodriguez. Before we begin, may I have your name and date of birth for verification? [2026-05-03 11:52:20] EXTERNAL: It's Sarah Johnson, date of birth 01/03/1974. [2026-05-03 11:52:40] INTERNAL: Thank you. What symptoms are you ex"
46b0c539-6958-4fde-9459-6abf0a95d78f,Kevin Rodriguez,Billing,2026-05-29 18:29:00,2026-05-29 18:39:49,649,15,"[2026-05-29 18:29:00] INTERNAL: HCP billing department, this is Kevin Rodriguez. How can I assist you? [2026-05-29 18:29:43] EXTERNAL: I have a question about a bill I received. I don't recognize this charge. [2026-05-29 18:30:26] INTERNAL: I understand your concern. Let me look into that for you. C"
d8990983-b298-4b6e-a447-3e49e3e4d09d,Robert Park,Pharmacy,2026-05-03 13:24:00,2026-05-03 13:27:29,209,15,"[2026-05-03 13:24:00] INTERNAL: HCP billing department, this is Robert Park. How can I assist you? [2026-05-03 13:24:13] EXTERNAL: I have a question about a bill I received. My copay should only be $25. [2026-05-03 13:24:27] INTERNAL: I understand your concern. Let me look into that for you. Can I g"
051ba86d-d1b5-4d5b-830d-d148a1b36599,Robert Nguyen,Referrals,2026-05-28 04:22:00,2026-05-28 04:25:08,188,15,"[2026-05-28 04:22:00] INTERNAL: HCP referrals, this is Robert Nguyen. How can I help you? [2026-05-28 04:22:12] EXTERNAL: I need a referral to see a endocrinologist. My doctor said they would send one but I haven't heard anything. [2026-05-28 04:22:25] INTERNAL: I can look into that. May I have your"


## Silver Layer Complete

**What we did:** Transformed 500+ individual utterances into 50 complete call transcripts (2-17 lines each), joined with Genesys metadata.

**Output schema (`silver_conversations`):**

| Column | Source | Purpose |
|--------|--------|---------|
| `full_transcript` | Stitched from `transcripts_raw` | AI scoring input -- with per-utterance timestamps inline (if available) |
| `transcript_start_time` | `transcripts_raw.transcript_start_time` | When the call began (may be NULL in some exports) |
| `transcript_end_time` | `transcripts_raw.transcript_end_time` | When the call ended (may be NULL in some exports) |
| `transcript_duration` | `transcripts_raw.transcript_duration` | Total call length (from Genesys) |
| `agent_name` | `interactions_raw.Users` | Agent performance tracking |
| `queue` | `interactions_raw.Queue` | Department-level reporting |
| `call_duration_seconds` | `interactions_raw.Duration` | Call length context for scoring |
| `num_utterances` | COUNT of transcript lines | Conversation complexity indicator |
| `division` | `interactions_raw.Division` | Facility-level rollup |
| `skills`, `language` | `interactions_raw` | Routing and language context |

**Next notebook -->** `02_AI_Scoring` runs sentiment analysis, rubric scoring, and topic extraction on each conversation.

---
### For the pod
This is the step that maps raw Genesys exports into AI-ready format. When switching from synthetic to real Genesys data, only the table references at the top need to change -- the transform logic stays the same.

---
### Scaling to Production

This notebook uses `CREATE OR REPLACE TABLE` (full rebuild) which is fine for 50 calls. In production with 10K+ daily calls:

| Concern | Hackathon (now) | Production | How |
|---------|----------------|------------|-----|
| Trigger | Manual run | Incremental on new files | Auto Loader + Lakeflow Declarative Pipeline |
| Processing | Full rebuild every run | Only new/changed calls | Streaming Table with `APPEND` flow |
| Partitioning | None (50 rows) | By date | `CLUSTER BY (transcript_start_time)` via Liquid Clustering |
| Monitoring | Manual verify cell | Automated alerts | Expectations: `EXPECT (num_utterances > 0)`, row count checks |
| Latency | Minutes (batch) | Seconds (streaming) | Trigger.AvailableNow or continuous |

**Lakeflow version of this stitch (pseudo-code):**
```sql
CREATE OR REFRESH STREAMING TABLE silver_conversations
AS SELECT ... -- same stitch logic
FROM STREAM(read_files('/path/to/genesys/transcripts/'))
```

**Key point for judges:** The SQL logic is identical at any scale -- only the orchestration wrapper changes. This is a 10-minute migration from notebook to Lakeflow pipeline.